# A full training

Install the Transformers, Datasets, and Evaluate libraries to run this notebook.

In [1]:
from datasets import load_dataset
import torch
from tqdm.auto import tqdm
from transformers import AutoTokenizer,AutoModelForSequenceClassification,DataCollatorWithPadding,get_scheduler
from accelerate import Accelerator, notebook_launcher
from torch.utils.data import DataLoader

In [2]:
check_point="google-bert/bert-base-uncased"
tokenizer=AutoTokenizer.from_pretrained(check_point)

In [4]:
dataset=load_dataset("nyu-mll/glue","mrpc")
dataset

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
def tokenizer_function(example):
  return tokenizer(example["sentence1"],example["sentence2"],truncation=True)
tokenized_dataset=dataset.map(tokenizer_function,batched=True)
data_collator=DataCollatorWithPadding(tokenizer=tokenizer)

Map:   0%|          | 0/3668 [00:00<?, ? examples/s]

Map:   0%|          | 0/408 [00:00<?, ? examples/s]

Map:   0%|          | 0/1725 [00:00<?, ? examples/s]

In [6]:
tokenized_dataset

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

In [8]:
tokenized_dataset = tokenized_dataset.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_dataset = tokenized_dataset.rename_column("label", "labels")
tokenized_dataset.set_format("torch") # Removed this line
tokenized_dataset["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [9]:
train_laoder=DataLoader(
    tokenized_dataset["train"],shuffle=True,batch_size=8, collate_fn=data_collator
)
eval_loader=DataLoader(
    tokenized_dataset["validation"],shuffle=True,batch_size=8, collate_fn=data_collator
)

In [10]:
model=AutoModelForSequenceClassification.from_pretrained(check_point,num_labels=2)
device = torch.device("cuda" if torch.accelerator.is_available() else "cpu")
model.to(device)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google-bert/bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [11]:
import torch.optim as optim
optimizer=optim.AdamW(model.parameters(),lr=3e-5)
num_epochs=3
num_training_steps=num_epochs*len(train_laoder)
lr_scheduler=get_scheduler(
    "linear",
    optimizer=optimizer,
    num_training_steps=num_training_steps,
    num_warmup_steps=0
)

In [12]:
# !pip install --upgrade datasets

In [15]:
acc=Accelerator()
train_dl,eval_dl,model,optimizer=acc.prepare(train_laoder,eval_loader,model,optimizer)

progress_bar=tqdm(range(num_training_steps))

model.train()

for epoch in range(num_epochs):

  for batch in train_dl:

    outputs=model(**batch)

    loss=outputs.loss

    acc.backward(loss)

    optimizer.step()

    lr_scheduler.step()

    optimizer.zero_grad()

    progress_bar.update(1)

  0%|          | 0/1377 [00:00<?, ?it/s]

In [20]:
import evaluate
metric=evaluate.load("glue","mrpc")
model.eval()
for batch in eval_dl:
  with torch.no_grad():
    outputs=model(**batch)
    loss=outputs.logits
    pred=torch.argmax(loss,dim=-1)
    metric.add_batch(predictions=pred,references=batch["labels"])
metric.compute()

{'accuracy': 0.8774509803921569, 'f1': 0.9143835616438356}